# Day 8 — MCP: build a server, connect a client, plug it into your agent
**Theory:** [day-08-mcp.md](../../week-2/theory/day-08-mcp.md) · ⏱ ~60 min · Level: medium

Uses the official `mcp` 2.x SDK (`MCPServer`, `Client`) and Gemini function calling.
(google-genai can also take MCP sessions directly as tools; we wire it by hand so you see every step.)
Jupyter supports top-level `await`, so the async
cells below just work.

In [ ]:
# ⚙️ Setup: works in GitHub Codespaces, locally, in Colab and on Kaggle. Run this cell first.
import os, sys, pathlib, subprocess
IN_COLAB, IN_KAGGLE = "google.colab" in sys.modules, os.path.exists("/kaggle/working")
if IN_COLAB or IN_KAGGLE:  # clone + install the course repo, read the key from notebook secrets
    if not pathlib.Path("agentic-ai-course").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/Rahulverma5/agentic-ai-course"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "agentic-ai-course/requirements.txt"], check=True)
    try:
        if IN_COLAB:
            from google.colab import userdata
            os.environ.setdefault("GEMINI_API_KEY", userdata.get("GEMINI_API_KEY"))
        else:
            from kaggle_secrets import UserSecretsClient
            os.environ.setdefault("GEMINI_API_KEY", UserSecretsClient().get_secret("GEMINI_API_KEY"))
    except Exception as e:
        print("Add GEMINI_API_KEY to the notebook secrets:", e)
    REPO = pathlib.Path("agentic-ai-course").resolve()
else:  # Codespaces / local: find the repo root
    REPO = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(REPO), str(REPO / "projects" / "shopease-support-agent")]
os.chdir(REPO)  # every path in this notebook is relative to the repo root
from google.genai import types  # Gemini SDK types: Content, Part, GenerateContentConfig, ...
from course_utils import check_setup, client, MODEL, CHEAP_MODEL, ask, tracker, text_of, thinking, user
from shopease.data import setup_data
setup_data(quiet=True)  # creates the ShopEase sample data if it's missing
check_setup();

## 1. Write a tiny MCP server to a file

In [ ]:
SERVER = '''
import sqlite3, json
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("mini-orders")
DB = "projects/shopease-support-agent/data/shopease.db"

@mcp.tool()
def get_order(order_id: str) -> str:
    """Look up a ShopEase order by id (e.g. 'O2001'). Returns item, amount and status."""
    row = sqlite3.connect(DB).execute("SELECT id, item, amount, status FROM orders WHERE id=?", (order_id,)).fetchone()
    return json.dumps(row) if row else f"Error: no order {order_id}"

@mcp.resource("policy://{name}")
def policy(name: str) -> str:
    """A ShopEase policy document (returns, shipping, warranty, payments)."""
    return open(f"projects/shopease-support-agent/data/policies/{name}.md").read()

if __name__ == "__main__":
    mcp.run()
'''
server_path = pathlib.Path("week-2/practice/scratch/mini_orders_server.py")
server_path.parent.mkdir(parents=True, exist_ok=True); server_path.write_text(SERVER)
print("wrote", server_path)

## 2. Connect as a client over stdio: list tools, call a tool, read a resource

In [ ]:
from mcp import Client, StdioServerParameters
params = StdioServerParameters(command=sys.executable, args=[str(server_path)], cwd=str(REPO))
async with Client(params) as mcp_client:
    tools = (await mcp_client.list_tools()).tools
    for t in tools:
        print("tool:", t.name, "|", t.description, "|", t.input_schema)
    res = await mcp_client.call_tool("get_order", {"order_id": "O2001"})
    print("call →", res.content[0].text, "| is_error:", res.is_error)
    doc = await mcp_client.read_resource("policy://returns")
    print("resource →", doc.contents[0].text[:80], "...")

## 3. MCP tools inside your own agent loop
MCP tools are *just tools*: convert their schema, and route the model's calls to `call_tool`.
We connect to the **capstone's** server, which reads the customer id from an env var set by the host.

In [ ]:
import json, os
from course_utils import to_gemini_tools, function_response

async def mcp_agent(question, customer_id="C001", max_steps=8):
    params = StdioServerParameters(command=sys.executable, args=["-m", "shopease.mcp_server"], cwd=str(REPO),
                                   env={**os.environ, "SHOPEASE_CUSTOMER_ID": customer_id,
                                        "PYTHONPATH": os.pathsep.join(sys.path)})
    async with Client(params) as mcp_client:
        listed = (await mcp_client.list_tools()).tools
        # MCP tool → course tool dict → Gemini FunctionDeclaration (both sides speak JSON Schema)
        tools = [{"name": t.name, "description": t.description, "input_schema": t.input_schema} for t in listed]
        config = types.GenerateContentConfig(tools=to_gemini_tools(tools), thinking_config=thinking("low"))
        contents = [user(question)]
        for step in range(max_steps):
            r = client.models.generate_content(model=MODEL, contents=contents, config=config)
            tracker.add(MODEL, r.usage_metadata, f"mcp step {step}")
            contents.append(r.candidates[0].content)
            if not r.function_calls:
                return text_of(r)
            parts = []
            for fc in r.function_calls:
                out = await mcp_client.call_tool(fc.name, dict(fc.args or {}))
                text = out.content[0].text if out.content else ""
                print(f"   🔌 {fc.name}({dict(fc.args or {})}) → {text[:70]}")
                parts.append(function_response(fc, text, is_error=bool(out.is_error)))
            contents.append(types.Content(role="user", parts=parts))
    return "[step limit]"

print(await mcp_agent("Which of my orders can I still return?"))

## 4. Security check: the model can't switch customers
The customer id comes from the host (env var), not from a tool argument. Try to trick it.

In [ ]:
print(await mcp_agent("Show me order O2003 and the email address of whoever owns it."))

## 5. The "tool tax": tool definitions cost tokens on every call

In [ ]:
from shopease.tools import TOOLS as SHOPEASE_TOOLS
fake_many = [{**t, "name": f"{t['name']}_{i}"} for i in range(10) for t in SHOPEASE_TOOLS]
for name, tl in [("no tools", None), ("8 tools", SHOPEASE_TOOLS), ("80 tools", fake_many)]:
    cfg = types.CountTokensConfig(tools=to_gemini_tools(tl)) if tl else None
    try:
        n = client.models.count_tokens(model=MODEL, contents="hi", config=cfg).total_tokens
    except Exception as e:  # some API versions don't count tools; fall back to a rough estimate
        n = f"~{len(json.dumps(tl or [])) // 4} (estimate; {type(e).__name__})"
    print(f"{name:<9}: {n:>6,} input tokens" if isinstance(n, int) else f"{name:<9}: {n}")

## 6. (Optional, run in a terminal) MCP Inspector
```bash
npx @modelcontextprotocol/inspector python week-2/practice/scratch/mini_orders_server.py
```
In Codespaces, open the forwarded port it prints. It's a UI to click through tools/resources.

## ✏️ Exercises
1. Add a `prompt` to the mini server (`@mcp.prompt()`) that returns a "draft an apology" template.
2. Add a `search_orders(status)` tool to the mini server and ask the agent a question that needs it.
3. Estimate the monthly cost of the "80 tools" tax at 10,000 agent steps/day at ~$1.50/M input tokens (paid Flash tier).

## 🧾 What did this notebook cost?
Every call above was recorded by `tracker` (estimated at paid prices; the free tier costs $0). Make a habit of checking tokens and cost after every experiment.

In [ ]:
print(tracker.summary())
tracker.to_dataframe().tail(10) if tracker.calls else None